# Section 3: Feature Engineering Visualizations

This notebook generates the required figures for Section 3 of the assignment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Import feature extraction function
from notebooks.SVM_Local_Training import extract_features_from_dataframe, load_data

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

## Figure 3.1: Raw Sensor Data Plot

Visualize raw IMU data from a single punch gesture

In [ ]:
# Load a sample punch gesture
sample_file = list(Path('../data/organized_training/multiclass_classification/punch').glob('*.csv'))[0]
df = pd.read_csv(sample_file)

# Create 6 subplots
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle('Raw IMU Data: Single Punch Gesture', fontsize=16, fontweight='bold')

# Time axis (relative to start)
time = (df['timestamp_ms'] - df['timestamp_ms'].iloc[0]) / 1000  # Convert to seconds

# Plot accelerometer data
for i, axis in enumerate(['accel_x', 'accel_y', 'accel_z']):
    ax = axes[i // 2, i % 2]
    ax.plot(time, df[axis], linewidth=2, color=f'C{i}')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Acceleration (m/s²)')
    ax.set_title(f'{axis.upper()}: Linear Acceleration', fontweight='bold')
    ax.grid(True, alpha=0.3)

# Plot gyroscope data
for i, axis in enumerate(['gyro_x', 'gyro_y', 'gyro_z']):
    ax = axes[(i+3) // 2, (i+3) % 2]
    ax.plot(time, df[axis], linewidth=2, color=f'C{i+3}')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Angular Velocity (rad/s)')
    ax.set_title(f'{axis.upper()}: Gyroscope', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assignment/figures/figure_3_1_raw_sensor_data.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Figure 3.1 saved: Raw sensor data from {sample_file.name}")
print(f"Duration: {time.iloc[-1]:.2f} seconds")
print(f"Samples: {len(df)}")

## Figure 3.2: Feature Distribution Comparison

In [ ]:
# Load multiclass data
classes = ['jump', 'punch', 'turn_left', 'turn_right', 'idle', 'noise']
X_multi, y_multi, feature_names = load_data(
    '../data/organized_training/multiclass_classification',
    classes
)

# Select 4 interesting features
interesting_features = [
    'accel_x_std',      # Separates idle (low) from active (high)
    'gyro_z_max',       # Separates turns (high) from straight (low)
    'accel_y_mean',     # Separates orientations
    'accel_x_fft_max'   # Separates periodic (walk) from ballistic (punch)
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
fig.suptitle('Feature Distributions by Gesture Class', fontsize=16, fontweight='bold')

colors = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5']

for i, feature_name in enumerate(interesting_features):
    if feature_name not in feature_names:
        print(f"Warning: {feature_name} not in features")
        continue
    
    feature_idx = feature_names.index(feature_name)
    
    for class_idx, class_name in enumerate(classes):
        mask = y_multi == class_idx
        data = X_multi[mask, feature_idx]
        axes[i].hist(data, bins=20, alpha=0.5, label=class_name, color=colors[class_idx])
    
    axes[i].set_xlabel(feature_name.replace('_', ' ').title())
    axes[i].set_ylabel('Count')
    axes[i].legend(loc='best', fontsize=9)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../assignment/figures/figure_3_2_feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure 3.2 saved: Feature distributions across classes")

## Figure 3.3: Feature Correlation Matrix

In [ ]:
# Compute correlation matrix
corr_matrix = np.corrcoef(X_multi.T)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr_matrix, 
            xticklabels=feature_names, 
            yticklabels=feature_names,
            cmap='coolwarm', 
            center=0, 
            vmin=-1, 
            vmax=1,
            cbar_kws={'label': 'Correlation'},
            ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../assignment/figures/figure_3_3_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure 3.3 saved: Feature correlation matrix")

## Figure 3.4: Class Distribution Bar Chart

In [ ]:
# Count samples per class
unique, counts = np.unique(y_multi, return_counts=True)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.bar([classes[i] for i in unique], counts, color='steelblue', edgecolor='black', linewidth=1.5)

# Add count labels on bars
for i, (idx, count) in enumerate(zip(unique, counts)):
    ax.text(i, count + 2, str(count), ha='center', fontweight='bold', fontsize=12)

ax.set_xlabel('Gesture Class', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=13, fontweight='bold')
ax.set_title('Multiclass Dataset: Sample Distribution', fontsize=16, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=15)

plt.tight_layout()
plt.savefig('../assignment/figures/figure_3_4_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure 3.4 saved: Class distribution")
print(f"Total samples: {sum(counts)}")

In [ ]:
print("\n" + "="*50)
print("All Section 3 figures generated successfully!")
print("="*50)
print("\nSaved to: assignment/figures/")
print("  - figure_3_1_raw_sensor_data.png")
print("  - figure_3_2_feature_distributions.png")
print("  - figure_3_3_correlation_matrix.png")
print("  - figure_3_4_class_distribution.png")